In [1]:
import os
import cv2
import numpy as np 
import matplotlib.pyplot as plt

from zipfile import ZipFile
from urllib.request import urlretrieve

%matplotlib inline


In [2]:
# if not os.path.isdir('models'):
#     os.mkdir("models")

# if not os.path.isfile("models"):
#     os.chdir("models")
#     # Download the tensorflow Model
#     urlretrieve('http://download.tensorflow.org/models/object_detection/ssd_mobilenet_v2_coco_2018_03_29.tar.gz', 'ssd_mobilenet_v2_coco_2018_03_29.tar.gz')

#     # Uncompress the file
#     !tar -xvf ssd_mobilenet_v2_coco_2018_03_29.tar.gz

#     # Delete the tar.gz file
#     os.remove('ssd_mobilenet_v2_coco_2018_03_29.tar.gz')

#     # Come back to the previous directory
#     os.chdir("..")

In [3]:
# def download_and_unzip(url, save_path):
#     print(f"Downloading and extracting assests....", end="")

#     # Downloading zip file using urllib package.
#     urlretrieve(url, save_path)

#     try:
#         # Extracting zip file using the zipfile package.
#         with ZipFile(save_path) as z:
#             # Extract ZIP file contents in the same directory.
#             z.extractall(os.path.split(save_path)[0])

#         print("Done")

#     except Exception as e:
#         print("\nInvalid file.", e)

In [4]:
# URL = r"https://www.dropbox.com/s/xoomeq2ids9551y/opencv_bootcamp_assets_NB13.zip?dl=1"

# asset_zip_path = os.path.join(os.getcwd(), f"opencv_bootcamp_assets_NB13.zip")

# # Download if assest ZIP does not exists. 
# if not os.path.exists(asset_zip_path):
#     download_and_unzip(URL, asset_zip_path)   

In [5]:
os.chdir("models/models/models")
os.getcwd()

'D:\\MY WORK\\AeroNITK\\opencv-bootcamp\\models\\models\\models'

In [6]:
classFile = "coco_class_labels.txt"

In [7]:
with open(classFile, "r") as file:
    labels = file.read().split("\n")

print(labels)

['unlabeled', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'street sign', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'hat', 'backpack', 'umbrella', 'shoe', 'eye glasses', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'plate', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'mirror', 'dining table', 'window', 'desk', 'toilet', 'door', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'blender', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush', 'hair brush', '']


In [8]:
modelFile = os.path.join("models", "ssd_mobilenet_v2_coco_2018_03_29", "frozen_inference_graph.pb")
configFile = os.path.join("models", "ssd_mobilenet_v2_coco_2018_03_29.pbtxt")

# Loading the model

In [9]:
net = cv2.dnn.readNetFromTensorflow(modelFile, configFile)

# detect objects 

In [10]:
def detect_objects(net, frame, dim = 300):
    blob = cv2.dnn.blobFromImage(frame, 1.0, size =  (dim, dim), mean = (0,0,0), swapRB = True, crop = False)
    net.setInput(blob)
    detected_objs = net.forward()
    return detected_objs

In [15]:
source = 0

cap = cv2.VideoCapture(source)
frame_width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
frame_height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
conf_threshold = 0.7
win_name = "Detect Objects"


cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
while cv2.waitKey(1) != 27:
    has_frame , frame = cap.read()
    if not has_frame:
        break

    # flipping the frame for convention
    frame = cv2.flip(frame, 1)

    objects = detect_objects(net, frame, 300)
    for i in range(objects.shape[2]):
        confidence = objects[0,0,i, 2]
        if confidence > conf_threshold:
            x_top_left = int(objects[0,0,i,3]*frame_width)
            y_top_left = int(objects[0,0,i,4]*frame_height)
            x_bottom_right = int(objects[0,0,i,5]*frame_width)
            y_bottom_right = int(objects[0,0,i,6]*frame_height)
            
            cv2.rectangle(frame, (x_top_left, y_top_left), (x_bottom_right, y_bottom_right), (0, 255, 255), thickness= 2, lineType = cv2.LINE_8)

            # putting the class label
            classId = int(objects[0,0,i,1])
            label = "Confidence{:.2f}  ||  {}"
            label = label.format(confidence,labels[classId].upper())
            label_size , base_line = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            label_height = label_size[1] + base_line
            label_width = label_size[0]
            cv2.rectangle(frame, (x_top_left, y_top_left-label_height),(x_top_left+label_width, y_top_left), (255,255,255), -1)
            cv2.putText(frame, label,(x_top_left, y_top_left-base_line), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0),1)
    t, _ = net.getPerfProfile()
    fps = (t*1000)/cv2.getTickFrequency()
    cv2.putText(frame, f"FPS:{fps}",(0,50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,0,0),1)
    cv2.imshow(win_name, frame)
cap.release()

cv2.destroyWindow(win_name)

In [17]:
! git checkout -b module-13

Switched to a new branch 'module-13'


In [26]:
! git status

On branch module-13
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   module-12.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/
	Untitled.ipynb
	abc.mp4
	goturn.caffemodel
	goturn.prototxt
	ldr-drago.jpg
	ldr-mantiuk.jpg
	ldr-reinhard.jpg
	models/
	nanotrack_backbone_sim.onnx
	nanotrack_head_sim.onnx
	opencv_bootcamp_assets_12.zip
	opencv_bootcamp_assets_NB11.zip
	race-carBOOSTING.mp4
	race-carCSRT.mp4
	race-carGOTURN.mp4
	race-carKCF.mp4
	race-carMEDIANFLOW.mp4
	race-carMIL.mp4
	race-carMOSSE.mp4
	race-carTLD.mp4
	race_car_out.avi
	race_car_out.mp4

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
! git add 